# 5.5 — Decision Tree Regression
### Splitting Data into Regions and Predicting the Mean

---

## From Classification to Regression — What Changes?

You already know Decision Tree Classifiers from Chapter 4. The structure is identical here. The only things that change are:

| | Decision Tree Classifier | Decision Tree Regressor |
|---|---|---|
| **Leaf output** | Most common class label | **Mean of all values in that leaf** |
| **Splitting criterion** | Gini impurity reduction | **MSE reduction** |
| **Overfitting fix** | max_depth, min_samples_leaf | Same |
| **Everything else** | Same | Same |

Same tree. Same exhaustive split search. Same pruning. Just different output and different split quality measure.

---

## The Core Idea — Regions and Means

Instead of drawing a line through data (like Linear Regression), a Decision Tree Regressor **divides the feature space into rectangular regions** and predicts the **mean value** of training examples in each region.

Example — predicting salary from experience:

```
Is experience < 3?
├── Yes → Is education > 1?
│         ├── Yes → Predict ₹8.2L  (mean of all PG freshers in training)
│         └── No  → Predict ₹6.1L  (mean of all UG freshers in training)
└── No  → Is experience < 7?
          ├── Yes → Predict ₹14.5L (mean of mid-level in training)
          └── No  → Predict ₹22.8L (mean of senior in training)
```

Each leaf = one region = one predicted number (the mean of that region's training values).

---

## How Splitting Works — MSE Reduction

At every node, the tree asks: *"which feature and which threshold, when I split here, reduces MSE the most?"*

**The exhaustive search process (what runs in background):**

For every feature:
  - Sort all unique values of that feature
  - Try every unique value as a threshold
  - For each threshold, split data into left (≤ threshold) and right (> threshold)
  - Calculate weighted MSE after split:

$$\text{MSE}_{split} = \frac{n_{left}}{n} \cdot MSE_{left} + \frac{n_{right}}{n} \cdot MSE_{right}$$

  - MSE of each child = variance of target values in that child
  - Pick the (feature, threshold) pair that gives the lowest weighted MSE

**MSE of a node** = how spread out the target values are:
$$MSE_{node} = \frac{1}{n}\sum(y_i - \bar{y})^2$$
where $\bar{y}$ is the mean of that node. A node with all identical values has MSE=0 (pure). A node with widely spread values has high MSE (impure).

---

## Why Decision Tree Handles Non-Linear Relationships Naturally

Linear Regression says: *"every extra year of experience adds the same fixed ₹X to salary."* One straight line for everyone.

Real world salary:
- 0→3 years: big jump (fresher to junior)
- 3→7 years: even bigger jump (junior to senior)
- 7→15 years: slows down (senior plateau)

Decision Tree handles this by splitting at experience=3 and experience=7 — each region gets its own mean prediction. **No equation needed. No feature engineering needed.**

---

## Overfitting — Same Problem, Same Fix

Unlimited tree → every leaf has 1 training example → predicts that exact value → training MSE = 0 → test MSE = terrible.

Fix:
- **`max_depth`** — limits tree depth. Each level = one more split. Shallow tree = simpler model.
- **`min_samples_leaf`** — each leaf must have at least this many training examples. Forces the mean to be based on enough data points to be reliable.

Best values found via **cross-validation** — never using the test set.

---

## Interpretability — The Biggest Strength

You can print the entire tree and explain every prediction in plain language. No black box. Ravi can show management exactly why an employee gets a certain predicted salary — which splits led to that leaf, what the criteria were.

This is something Linear Regression also has (coefficients), but Polynomial Regression loses (too many engineered features to interpret).

---

## Real World Problem — Hyderabad Tech Employee Salary Prediction

**Ravi** is an HR manager at a tech company in Hyderabad. The company has grown rapidly and Ravi needs a fair, transparent salary prediction system for new hires and appraisals.

Features:
- `years_experience` — total years of work experience
- `age` — employee age
- `num_skills` — number of technical skills
- `previous_companies` — number of companies worked at
- `education_level` — 1 (UG), 2 (PG), 3 (PhD)

Target: `salary_lakhs` (annual CTC in ₹ lakhs)

The salary-experience relationship has natural breakpoints — fresher, junior, mid-level, senior. Decision Tree captures these automatically.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, export_text, plot_tree
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# WHY each import:
# DecisionTreeRegressor — sklearn's regression tree (uses CART algorithm internally)
# export_text           — prints the tree as readable text rules
# plot_tree             — draws the tree visually
# GridSearchCV          — tries all combinations of hyperparameters using CV
#                         finds best max_depth + min_samples_leaf together
# cross_val_score       — runs k-fold CV for a given model, returns scores per fold
# LinearRegression      — baseline comparison

np.random.seed(42)

In [ ]:
# ── Step 1: Create Dataset ────────────────────────────────────────────────────
n = 700

years_exp          = np.random.randint(0, 20, n)
age                = years_exp + np.random.randint(21, 26, n)  # age = exp + starting age
num_skills         = np.random.randint(2, 15, n)
previous_companies = np.random.randint(0, 6, n)
education_level    = np.random.choice([1, 2, 3], n, p=[0.5, 0.4, 0.1])
# WHY p=[0.5, 0.4, 0.1]?
# Realistic distribution — most employees are UG, fewer PG, even fewer PhD

# TRUE salary formula — non-linear with experience (has natural breakpoints)
noise = np.random.normal(0, 2, n)

# Salary grows fast early, then plateaus — captured by sqrt(experience)
salary_lakhs = (
    4.0  * np.sqrt(years_exp + 1)    +  # non-linear experience effect
    0.3  * num_skills                +  # more skills = more pay
    0.5  * previous_companies        +  # job hopping = higher pay (Indian market)
    2.0  * education_level           +  # higher education = higher base
    0.1  * (age - 25).clip(0)        +  # slight age premium
    5.0                              +  # base salary
    noise
).clip(3, 60)  # realistic salary range ₹3L to ₹60L

df = pd.DataFrame({
    'years_experience':   years_exp,
    'age':                age,
    'num_skills':         num_skills,
    'previous_companies': previous_companies,
    'education_level':    education_level,
    'salary_lakhs':       np.round(salary_lakhs, 2)
})

print(f"Dataset shape: {df.shape}")
print(f"Salary range: ₹{df['salary_lakhs'].min()}L to ₹{df['salary_lakhs'].max()}L")
print(f"Average salary: ₹{df['salary_lakhs'].mean():.2f}L")
df.head()

In [ ]:
# ── Step 2: Visualise the Non-Linear Relationship ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['years_experience'], df['salary_lakhs'],
                alpha=0.3, color='steelblue', s=15)
axes[0].set_xlabel('Years of Experience')
axes[0].set_ylabel('Salary (₹ lakhs)')
axes[0].set_title('Experience vs Salary\n(Non-linear — fast growth early, plateau later)')

axes[1].scatter(df['num_skills'], df['salary_lakhs'],
                alpha=0.3, color='tomato', s=15)
axes[1].set_xlabel('Number of Skills')
axes[1].set_ylabel('Salary (₹ lakhs)')
axes[1].set_title('Skills vs Salary\n(More linear — each skill adds similar value)')

plt.tight_layout()
plt.show()

# WHY visualise?
# Experience vs Salary shows a curve — confirms Decision Tree is appropriate.
# Linear Regression would draw one straight line and systematically underpredict
# for mid-level employees and overpredict for very senior ones.

In [ ]:
# ── Step 3: Split Data ────────────────────────────────────────────────────────
X = df.drop('salary_lakhs', axis=1)
y = df['salary_lakhs']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training: {X_train.shape[0]} employees | Test: {X_test.shape[0]} employees")

In [ ]:
# ── Step 4: DecisionTreeRegressor — What Happens Inside .fit() ────────────────
#
# sklearn's DecisionTreeRegressor uses the CART algorithm internally:
#
# CART = Classification And Regression Trees
#
# What .fit() does step by step:
#   1. Start with all training data at the root node
#   2. For every feature:
#        Sort unique values of that feature
#        For every unique value as threshold:
#          Split data: left = rows where feature <= threshold
#                      right = rows where feature > threshold
#          Calculate weighted MSE:
#            MSE_split = (n_left/n)*MSE_left + (n_right/n)*MSE_right
#          where MSE_left = variance of y values in left child
#                MSE_right = variance of y values in right child
#   3. Pick the (feature, threshold) with lowest MSE_split
#   4. Split the node. Recurse on each child.
#   5. Stop when: max_depth reached, OR node has < min_samples_split rows,
#                 OR all y values in node are identical (MSE=0)
#   6. Each leaf stores: mean of y values of all training rows that fell there
#
# What .predict() does:
#   For each new row: traverse the tree top to bottom
#   At each node: check if feature <= threshold → go left, else go right
#   Reach a leaf → return that leaf's stored mean value
#   Time complexity: O(depth) per sample

# First — unlimited tree (no constraints) to show overfitting
dt_unlimited = DecisionTreeRegressor(
    random_state=42
    # No max_depth, no min_samples_leaf
    # Tree will grow until every leaf has exactly 1 training sample
)
dt_unlimited.fit(X_train, y_train)
# .fit() runs the full CART algorithm described above

train_pred_unl = dt_unlimited.predict(X_train)
test_pred_unl  = dt_unlimited.predict(X_test)

train_rmse_unl = np.sqrt(mean_squared_error(y_train, train_pred_unl))
test_rmse_unl  = np.sqrt(mean_squared_error(y_test,  test_pred_unl))

print("Unlimited Tree (no constraints):")
print(f"  Tree depth reached : {dt_unlimited.get_depth()}")
print(f"  Number of leaves   : {dt_unlimited.get_n_leaves()}")
print(f"  Train RMSE         : ₹{train_rmse_unl:.4f}L  ← nearly zero (memorised everything)")
print(f"  Test RMSE          : ₹{test_rmse_unl:.2f}L   ← terrible on new data")
print(f"\nThis is pure overfitting — the tree memorised {X_train.shape[0]} training examples")
print(f"by creating {dt_unlimited.get_n_leaves()} leaf nodes — almost one per training row.")

In [ ]:
# ── Step 5: From Scratch — MSE Split Calculation ──────────────────────────────
# Manually implement what the tree does at ONE node
# to show exactly how it picks the best split

def mse_of_node(y_values):
    """MSE of a node = variance of target values in that node.
    MSE = (1/n) * sum((y - mean(y))^2)
    A pure node (all same value) has MSE=0.
    A messy node (widely spread values) has high MSE.
    """
    if len(y_values) == 0:
        return 0
    mean_y = np.mean(y_values)
    return np.mean((y_values - mean_y) ** 2)

def find_best_split(X_col, y):
    """Find the best threshold for one feature.
    This is what sklearn runs for EVERY feature at EVERY node.
    """
    best_threshold = None
    best_mse       = float('inf')
    n              = len(y)

    # Try every unique value as a threshold
    thresholds = np.unique(X_col)

    for threshold in thresholds:
        # Split into left and right
        left_mask  = X_col <= threshold
        right_mask = X_col >  threshold

        y_left  = y[left_mask]
        y_right = y[right_mask]

        if len(y_left) == 0 or len(y_right) == 0:
            continue  # skip splits that put everything on one side

        # Weighted MSE after split
        mse_split = (
            (len(y_left)  / n) * mse_of_node(y_left) +
            (len(y_right) / n) * mse_of_node(y_right)
        )

        if mse_split < best_mse:
            best_mse       = mse_split
            best_threshold = threshold

    return best_threshold, best_mse

# Run on the training data — find best split across all features
print("Manual split search at root node:")
print(f"Root MSE (before any split): {mse_of_node(y_train.values):.4f}")
print()

results = []
for col in X_train.columns:
    threshold, mse = find_best_split(X_train[col].values, y_train.values)
    results.append((col, threshold, mse))
    print(f"  {col:<22}: best threshold = {threshold:>6}, weighted MSE after split = {mse:.4f}")

best_col, best_thresh, best_mse = min(results, key=lambda x: x[2])
print(f"\nBest split: '{best_col}' <= {best_thresh}")
print(f"This matches what sklearn's DecisionTreeRegressor will pick at the root.")

In [ ]:
# ── Step 6: Effect of max_depth ───────────────────────────────────────────────
depths      = [1, 2, 3, 5, 7, 10, None]  # None = unlimited
train_rmses = []
test_rmses  = []

for depth in depths:
    dt = DecisionTreeRegressor(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    train_rmses.append(np.sqrt(mean_squared_error(y_train, dt.predict(X_train))))
    test_rmses.append(np.sqrt(mean_squared_error(y_test,  dt.predict(X_test))))

depth_labels = [str(d) if d is not None else 'None\n(unlimited)' for d in depths]

plt.figure(figsize=(10, 4))
plt.plot(range(len(depths)), train_rmses, 'o-', color='steelblue',
         label='Train RMSE', linewidth=2)
plt.plot(range(len(depths)), test_rmses,  'o-', color='tomato',
         label='Test RMSE',  linewidth=2)
plt.xticks(range(len(depths)), depth_labels)
plt.xlabel('max_depth')
plt.ylabel('RMSE (₹ lakhs)')
plt.title('Effect of max_depth — Finding the Sweet Spot')
plt.legend()
plt.tight_layout()
plt.show()

print("\ndepth | Train RMSE | Test RMSE")
print("-" * 35)
for d, tr, te in zip(depth_labels, train_rmses, test_rmses):
    print(f"  {str(d):<8} | ₹{tr:>6.3f}L  | ₹{te:>6.3f}L")

# WHY this plot?
# depth=1: too simple — underfits (high train AND test RMSE)
# depth=3-5: sweet spot — train and test RMSE both reasonable
# depth=None: train RMSE ≈ 0 but test RMSE explodes — pure overfitting

In [ ]:
# ── Step 7: GridSearchCV — Finding Best Hyperparameters ───────────────────────
#
# GridSearchCV works as follows internally:
#
# 1. Takes a grid of hyperparameter combinations
#    e.g. max_depth=[2,3,4,5] × min_samples_leaf=[2,5,10,20] = 16 combinations
#
# 2. For EACH combination:
#    Runs k-fold cross-validation on the training data:
#      Split train into k folds
#      For each fold: train on k-1 folds, evaluate on 1 fold
#      Average the k scores
#
# 3. Picks the combination with the best average CV score
#
# 4. Retrains on the FULL training data with the best combination
#    Stores as .best_estimator_
#
# WHY GridSearchCV and not just try depths one by one?
# Two hyperparameters interact — the best max_depth CHANGES depending on
# min_samples_leaf. You need to search them TOGETHER, not separately.

param_grid = {
    'max_depth':        [2, 3, 4, 5, 6, 7],
    'min_samples_leaf': [2, 5, 10, 20, 30]
}
# This creates 6×5 = 30 combinations
# Each combination gets 5-fold CV = 150 model fits total

grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,                              # 5-fold CV
    scoring='neg_root_mean_squared_error',  # negative RMSE (higher = better in sklearn)
    n_jobs=-1                          # use all CPU cores — runs combinations in parallel
)
# WHY n_jobs=-1?
# -1 tells sklearn to use ALL available CPU cores.
# 150 fits can run in parallel — much faster than sequential.

grid_search.fit(X_train, y_train)
# .fit() runs ALL 150 combinations, picks best, retrains on full X_train

print("GridSearchCV Results:")
print(f"  Best max_depth        : {grid_search.best_params_['max_depth']}")
print(f"  Best min_samples_leaf : {grid_search.best_params_['min_samples_leaf']}")
print(f"  Best CV RMSE          : ₹{-grid_search.best_score_:.4f}L")
# WHY negate best_score_?
# GridSearchCV stores neg_root_mean_squared_error — we negate to get positive RMSE

In [ ]:
# ── Step 8: Evaluate Best Model ───────────────────────────────────────────────
best_dt = grid_search.best_estimator_
# .best_estimator_ is the model retrained on full X_train with best hyperparameters
# This is what you use for predictions — not grid_search itself

y_pred     = best_dt.predict(X_test)
test_rmse  = np.sqrt(mean_squared_error(y_test, y_pred))
test_mae   = mean_absolute_error(y_test, y_pred)
test_r2    = r2_score(y_test, y_pred)

# mean_absolute_error — WHY use MAE here alongside RMSE?
# RMSE penalises large errors heavily (squared)
# MAE is more intuitive — "on average predictions are off by ₹X lakhs"
# For salary prediction, MAE is easier for Ravi to explain to management

print(f"Best Decision Tree Regressor:")
print(f"  max_depth        : {best_dt.max_depth}")
print(f"  min_samples_leaf : {best_dt.min_samples_leaf}")
print(f"  Tree depth used  : {best_dt.get_depth()}")
print(f"  Number of leaves : {best_dt.get_n_leaves()}")
print(f"\nTest Performance:")
print(f"  RMSE : ₹{test_rmse:.2f} lakhs")
print(f"  MAE  : ₹{test_mae:.2f} lakhs  ← on average off by this much")
print(f"  R²   : {test_r2:.4f}  ← explains {test_r2*100:.1f}% of salary variation")

In [ ]:
# ── Step 9: Visualise the Tree — Full Interpretability ────────────────────────

# Text representation — readable rules
print("Decision Tree Rules (Text):")
print("=" * 60)
tree_rules = export_text(
    best_dt,
    feature_names=list(X.columns)
)
# export_text internally traverses the tree and formats each node as:
# |--- feature <= threshold  (left branch)
# |--- feature >  threshold  (right branch)
# |--- value: X.XX           (leaf node — this is the mean prediction)
print(tree_rules)

# WHY export_text?
# Ravi can read this and explain to any employee:
# "Your salary is ₹X because you have Y years experience and Z skills"
# No black box — complete transparency.

In [ ]:
# ── Step 10: Visual Tree Plot ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    best_dt,
    feature_names=list(X.columns),
    filled=True,          # WHY filled=True? — colour nodes by prediction value
    rounded=True,         # rounded corners
    fontsize=8,
    ax=ax
)
# plot_tree internally:
# 1. Traverses the tree structure stored in best_dt.tree_
# 2. For each node: draws a box with feature name, threshold, MSE, n samples, mean value
# 3. Colours by mean value — darker = higher salary prediction
# 4. Draws edges between parent and child nodes

plt.title(f'Decision Tree Regressor — Depth {best_dt.get_depth()} | '
          f'Leaves: {best_dt.get_n_leaves()}', fontsize=12)
plt.tight_layout()
plt.show()

# Each box in the tree shows:
# Line 1: split condition (e.g. years_experience <= 3.5)
# Line 2: MSE of this node (how spread out salaries are here)
# Line 3: samples (how many training employees fell here)
# Line 4: value (mean salary = what the model predicts for this leaf)

In [ ]:
# ── Step 11: Feature Importance ───────────────────────────────────────────────
#
# Feature importance in Decision Tree Regressor:
# For each feature: sum up MSE reduction from ALL nodes that split on this feature
# Weighted by number of samples at each node
# Then normalised so all importances sum to 1.0
#
# Formula for one node split on feature f:
# importance_contribution = (n_node/n_total) * (MSE_parent - weighted_MSE_children)
#
# A feature that reduces MSE a lot across many samples = high importance
# A feature that is never used in any split = importance = 0.0

importances = best_dt.feature_importances_
# .feature_importances_ runs the above calculation internally after .fit()

feat_imp_df = pd.DataFrame({
    'Feature':    X.columns,
    'Importance': importances
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(8, 4))
plt.barh(feat_imp_df['Feature'], feat_imp_df['Importance'], color='steelblue')
plt.xlabel('Feature Importance (MSE reduction contribution)')
plt.title('Which Features Drive Salary Prediction?')
plt.tight_layout()
plt.show()

print("Feature importances (sum = 1.0):")
for feat, imp in zip(feat_imp_df['Feature'], feat_imp_df['Importance']):
    bar = '█' * int(imp * 50)
    print(f"  {feat:<22}: {imp:.4f}  {bar}")

In [ ]:
# ── Step 12: Compare with Linear Regression ───────────────────────────────────
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LinearRegression())
])
lr_pipe.fit(X_train, y_train)
lr_pred = lr_pipe.predict(X_test)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2   = r2_score(y_test, lr_pred)

print("=" * 55)
print("MODEL COMPARISON — HYDERABAD TECH SALARY")
print("=" * 55)
print(f"{'Model':<35} {'RMSE':>8} {'R²':>8}")
print("-" * 55)
print(f"{'Linear Regression':<35} ₹{lr_rmse:>5.2f}L  {lr_r2:>6.4f}")
print(f"{'Decision Tree (tuned)':<35} ₹{test_rmse:>5.2f}L  {test_r2:>6.4f}")
print("=" * 55)
improvement = ((lr_rmse - test_rmse) / lr_rmse) * 100
print(f"\nDecision Tree improved RMSE by {improvement:.1f}%")
print("Improvement comes from capturing the non-linear experience-salary curve.")

In [ ]:
# ── Step 13: Actual vs Predicted + Residuals ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred, alpha=0.4, color='steelblue', s=20)
axes[0].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()],
             'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Salary (₹ lakhs)')
axes[0].set_ylabel('Predicted Salary (₹ lakhs)')
axes[0].set_title('Decision Tree — Actual vs Predicted')
axes[0].legend()

residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.4, color='tomato', s=20)
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Predicted Salary (₹ lakhs)')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot\n(Staircase pattern = tree predicts discrete means)')

plt.tight_layout()
plt.show()

# WHY does residual plot look like vertical stripes/staircases?
# Decision Tree predicts DISCRETE values — one mean per leaf.
# If tree has 12 leaves, it can only predict 12 different values.
# Many test examples get the same prediction → vertical stripe of residuals.
# This is normal and expected for Decision Tree Regressor.
print("\nNote: The staircase/stripe pattern in residuals is EXPECTED for Decision Trees.")
print(f"This tree has {best_dt.get_n_leaves()} leaves = {best_dt.get_n_leaves()} possible predicted values.")
print("Many employees get the same prediction → vertical stripes.")

In [ ]:
# ── Step 14: Ravi's Salary Estimates for New Hires ────────────────────────────
new_hires = pd.DataFrame({
    'years_experience':   [0,  3,  7,  12, 18],
    'age':                [22, 26, 30, 36, 42],
    'num_skills':         [3,  6,  10, 12, 14],
    'previous_companies': [0,  1,  3,  4,  5],
    'education_level':    [1,  2,  2,  3,  2]
})

salary_estimates = best_dt.predict(new_hires)

labels = ['Fresher', 'Junior', 'Mid-level', 'Senior', 'Principal']
print("Ravi's Salary Estimates for New Hires:")
print("-" * 65)
print(f"{'Level':<12} {'Exp':>4} {'Skills':>7} {'Edu':>5} {'Est. Salary':>14}")
print("-" * 65)
for label, (_, row), sal in zip(labels, new_hires.iterrows(), salary_estimates):
    edu_map = {1: 'UG', 2: 'PG', 3: 'PhD'}
    print(f"{label:<12} {int(row['years_experience']):>3}yr "
          f"{int(row['num_skills']):>6} {edu_map[row['education_level']]:>6} "
          f"   ₹{sal:>6.2f} lakhs")

print("\nThe tree captured the non-linear salary growth:")
print("Fast rise in early years, gradual plateau at senior levels.")

---

## Internal Functions — Quick Reference

| Function / Attribute | What it does internally |
|---|---|
| `.fit(X, y)` | Runs CART algorithm — exhaustive split search using MSE reduction at every node |
| `.predict(X)` | Traverses tree top→bottom for each row, returns leaf's stored mean value |
| `.get_depth()` | Returns max depth of learned tree (not max_depth param — actual depth used) |
| `.get_n_leaves()` | Counts terminal nodes (leaves) in learned tree |
| `.feature_importances_` | Sum of weighted MSE reduction per feature across all nodes, normalised to 1.0 |
| `export_text()` | Traverses tree_ object, formats each node as text rule |
| `plot_tree()` | Traverses tree_ object, draws boxes and edges with matplotlib |
| `GridSearchCV.fit()` | Tries all param combinations × k folds, picks best, retrains on full train data |
| `.best_estimator_` | The model retrained with best params on full training data |
| `.best_params_` | Dictionary of the winning hyperparameter combination |
| `.best_score_` | Best CV score (neg RMSE here — negate to get positive RMSE) |

---

## Summary Table

| | Decision Tree Regressor |
|---|---|
| **Task** | Regression with non-linear relationships |
| **How it works** | Splits feature space into regions, predicts mean of each region |
| **Splitting criterion** | MSE reduction (weighted MSE of children vs parent) |
| **Leaf prediction** | Mean of all training y values that fell in that leaf |
| **Algorithm** | CART — tries every feature × every threshold |
| **Key hyperparameters** | `max_depth`, `min_samples_leaf` |
| **How to tune** | GridSearchCV — searches both together (they interact) |
| **Needs scaling?** | ❌ No — splits are based on thresholds, not distances or penalties |
| **Strength** | Non-linear, interpretable, no scaling needed, fast |
| **Weakness** | Unstable (small data change = different tree), limited accuracy |
| **Residual pattern** | Staircase/stripes — expected, because predictions are discrete means |
| **When to use** | Non-linear data, need interpretability, quick baseline |

---

## What's Next?

Decision Tree Regressor is interpretable but unstable and limited in accuracy.

**5.6 Random Forest Regression** fixes the instability by building hundreds of trees on different random subsets of data and averaging their predictions. Same bagging idea from Chapter 4 — but for regression. Errors cancel out. Accuracy improves dramatically.

---

## Practice Task

Meena manages a chain of restaurants in Bangalore. She wants to predict **daily revenue (₹ thousands)** based on:
- `day_of_week` — 1 (Mon) to 7 (Sun)
- `temperature_c` — outside temperature
- `is_holiday` — 0 or 1
- `num_staff` — number of staff on duty
- `nearby_events` — number of events happening nearby

Revenue has natural breakpoints — weekends spike, holidays behave differently, hot days affect footfall.

**Your tasks:**

1. Create synthetic dataset of 800 days with non-linear revenue patterns
2. Train unlimited Decision Tree — show overfitting (train vs test RMSE)
3. Use GridSearchCV to find best `max_depth` and `min_samples_leaf`
4. Print the tree rules using `export_text` — interpret them in plain English
5. Plot feature importances — which factor drives revenue most?
6. Compare with Linear Regression baseline

In [ ]:
# YOUR CODE HERE

# Step 1: Create dataset

# Step 2: Unlimited tree — show overfitting

# Step 3: GridSearchCV — best max_depth + min_samples_leaf

# Step 4: Print tree rules + interpret

# Step 5: Feature importances plot

# Step 6: Compare with Linear Regression